# 🏔️ Gradient Descent & Its Variants
### Batch ⚡ Stochastic 🎲 Mini-Batch 📦 — built from scratch with NumPy

---

## 📑 Table of Contents
1. 📘 Introduction — what is Gradient Descent?
2. 🧮 The math behind it
3. 🛠️ Setup — imports
4. 🌱 Warm-up — Gradient Descent on 1 feature
5. ⚡ Batch Gradient Descent (own class)
6. 🎲 Stochastic Gradient Descent (own class + scikit-learn)
7. 📦 Mini-Batch Gradient Descent (own class + scikit-learn)
8. ⚖️ Comparison of all three variants
9. 📝 Notes on code improvements
10. 🏁 Summary

## 📘 1. Introduction

**Gradient Descent (GD)** is an *optimization algorithm* used to find the values of model parameters (weights/coefficients) that **minimize a loss function**.

🏔️ **Intuition:** imagine you are standing on a foggy mountain and want to reach the valley. You cannot see the whole mountain, but you can feel the slope under your feet. So you repeatedly take a small step **downhill** (opposite to the slope). After enough steps you reach the bottom.

| Mountain world 🏔️ | Machine-learning world 🤖 |
|---|---|
| Your position | Model parameters (`m`, `b`, coefficients) |
| Height of the ground | Loss (error) value |
| Slope under your feet | Gradient of the loss |
| Size of your step | Learning rate (`lr`) |
| One step | One parameter update |

📌 **Where is it used?** Linear & logistic regression, neural networks, deep learning, SVMs and almost every modern ML model that is trained by minimizing an error.

There are **three popular variants**. They differ only in *how much data is used to compute the gradient for one update*:

- ⚡ **Batch GD** → all the samples
- 🎲 **Stochastic GD** → 1 sample
- 📦 **Mini-Batch GD** → a small group of samples

## 🧮 2. The math behind it

For linear regression the prediction is

$$\hat{y} = m\,x + b \qquad\text{(or } \hat{y} = X\cdot w + b \text{ for many features)}$$

and the loss is the **Mean Squared Error (MSE)**:

$$L = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2$$

Taking partial derivatives (the *gradients*):

$$\frac{\partial L}{\partial b} = -\frac{2}{n}\sum (y_i-\hat{y}_i) \qquad\qquad \frac{\partial L}{\partial w_j} = -\frac{2}{n}\sum (y_i-\hat{y}_i)\,x_{ij}$$

**Update rule** (repeated for every step):

$$\boxed{\theta_{new} = \theta_{old} - \eta \cdot \frac{\partial L}{\partial \theta}}$$

where **η (eta) = learning rate**.

⚠️ **Learning rate matters a lot:**
- 🐢 too small → very slow training
- 🚀 too large → overshoots the minimum and may diverge
- 🎯 just right → smooth, fast convergence

## 🛠️ 3. Setup — imports

All libraries are imported once here. We also fix the random seeds so the results are **reproducible** 🔁.

💡 We import scikit-learn's `SGDRegressor` with an **alias** (`SklearnSGDRegressor`) so it does not overwrite our own `SGDRegressor` class that we write later.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor as SklearnSGDRegressor   # alias -> avoids name clash with our own class
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, cross_val_score

# Reproducibility
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8-whitegrid")

## 🌱 4. Warm-up — Gradient Descent on 1 feature

Before using many features, let us understand the idea on the simplest case: **one input `x`, one output `y`**, with just two parameters — slope `m` and intercept `b`.

### 🔹 Step 1: Create a toy dataset
`make_regression` generates a noisy straight-line dataset. `noise=20` adds randomness so the points do not lie perfectly on a line.

In [ ]:
x1, y1 = make_regression(n_samples=100, n_features=1, n_informative=1,
                         n_targets=1, noise=20, random_state=42)

plt.scatter(x1, y1, color="teal", alpha=0.7)
plt.title("Toy regression data")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

### 🔹 Step 2: The "answer key" — scikit-learn's `LinearRegression`
`LinearRegression` solves the problem **exactly** (closed-form solution). Its slope and intercept are the targets our gradient descent should approach. 🎯  
We also compute a 10-fold cross-validated R² score as a benchmark.

In [ ]:
lr_model = LinearRegression().fit(x1, y1)
print("Slope (m)     :", lr_model.coef_[0])
print("Intercept (b) :", lr_model.intercept_)
print("CV R² score   :", np.mean(cross_val_score(lr_model, x1, y1, scoring="r2", cv=10)))

### 🔹 Step 3: Our own 1-feature Gradient Descent class

**How `fit` works, line by line:**

1. **Start** with deliberately bad guesses: `m = 100`, `b = -120`.
2. **Loop** `epochs` times. In every loop:
   - `err = y - (m*x + b)` → how wrong are we right now?
   - `slope_b` and `slope_m` → the gradients of the loss w.r.t. `b` and `m` (formulas from Section 2).
   - `b -= lr * slope_b` and `m -= lr * slope_m` → move **against** the gradient.
3. Both gradients are computed from the *old* values before either parameter is changed, so the update is correct.
4. `loss_history_` stores the MSE of every epoch so we can plot the learning curve 📉.

In [ ]:
class SimpleGDRegressor:
    def __init__(self, learning_rate=0.1, epochs=100):
        self.m = 100            # deliberately bad starting values
        self.b = -120
        self.lr = learning_rate
        self.epochs = epochs
        self.loss_history_ = []

    def fit(self, x, y):
        x = x.ravel()                                   # (n,1) -> (n,)
        for _ in range(self.epochs):
            err = y - (self.m * x + self.b)
            self.loss_history_.append(np.mean(err ** 2))
            slope_b = -2 * np.mean(err)                 # dL/db
            slope_m = -2 * np.mean(err * x)             # dL/dm
            self.b -= self.lr * slope_b
            self.m -= self.lr * slope_m
        return self

    def predict(self, x):
        return self.m * x.ravel() + self.b

In [ ]:
gd1 = SimpleGDRegressor(learning_rate=0.1, epochs=100).fit(x1, y1)
print(f"GD  -> m = {gd1.m:.4f}, b = {gd1.b:.4f}")
print(f"OLS -> m = {lr_model.coef_[0]:.4f}, b = {lr_model.intercept_:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x1, y1, color="teal", alpha=0.6, label="data")
xs = np.linspace(x1.min(), x1.max(), 100).reshape(-1, 1)
ax[0].plot(xs, gd1.predict(xs), color="crimson", lw=2, label="Gradient Descent line")
ax[0].set_title("Fitted line"); ax[0].legend()
ax[1].plot(gd1.loss_history_, color="darkorange")
ax[1].set_title("Loss (MSE) vs epoch"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("MSE")
plt.tight_layout(); plt.show()

✅ **Result:** starting from a terrible guess, gradient descent lands on (almost) the same slope and intercept as the exact solution, and the loss curve falls steeply and then flattens. The same idea scales to many features — that is what we do next.

---
## ⚡ 5. Batch Gradient Descent

### 📖 Definition
**Batch Gradient Descent** computes the gradient using the **entire training set** and then makes **one** parameter update. One pass over all the data = one epoch = one update.

### 🕒 When is it used?
- Small / medium datasets that fit in memory
- Convex problems (like linear regression) where you want a smooth, stable path to the minimum
- When you need deterministic, reproducible training

### ✅ Advantages
- 📉 Smooth and stable convergence (no noise in the gradient)
- 🎯 Guaranteed to reach the global minimum for convex loss (with a suitable learning rate)
- 🔁 Deterministic — same data gives same result
- ⚙️ Vectorizes very well

### ❌ Disadvantages
- 🐌 Very slow on large datasets (every update needs a full pass)
- 💾 Needs the whole dataset in memory
- 🪤 Can get stuck in local minima / saddle points for non-convex problems (e.g. neural networks)
- ❌ No online learning (cannot learn from streaming data)

### 📊 Dataset: Diabetes (442 patients, 10 features)
First we split the data and set a baseline with `LinearRegression`.

In [ ]:
x, y = load_diabetes(return_X_y=True)
print("Features:", x.shape, "| Target:", y.shape)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

reg = LinearRegression().fit(x_train, y_train)
r2_ols = r2_score(y_test, reg.predict(x_test))
print("Baseline LinearRegression R² :", r2_ols)

### 🧑‍💻 Our own `GDRegressor` class (Batch GD) — explained in depth

**`__init__`**  
Stores the hyper-parameters (`learning_rate`, `epochs`) and creates empty `coef_` and `intercept_`. Names with a trailing `_` follow the scikit-learn convention for *learned* values.

**`fit` — the heart of the algorithm**

| Line | What it does |
|---|---|
| `self.intercept_ = 0` / `self.coef_ = np.ones(n_features)` | Initial guess: intercept 0, every coefficient 1 |
| `y_hat = np.dot(x_train, self.coef_) + self.intercept_` | Predictions for **all** training rows at once (matrix–vector product) |
| `slope_intercept = -2 * np.mean(y_train - y_hat)` | Gradient of MSE w.r.t. the intercept |
| `coef_der = -2 * np.dot((y_train - y_hat), x_train) / n` | Gradient w.r.t. **all** coefficients in one shot. `(y-ŷ)` has shape `(n,)`, `x_train` is `(n, 10)` → result has shape `(10,)` — one gradient per feature |
| `self.intercept_ -= lr * slope_intercept` and `self.coef_ -= lr * coef_der` | Step downhill |

🔑 **Key point:** the gradient uses **all `n` rows** → exactly **1 update per epoch**.

**`predict`** — simply `X·w + b`.

📉 `loss_history_` records the training MSE of each epoch (used later for the comparison plot).

In [ ]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.loss_history_ = []

    def fit(self, x_train, y_train):
        # initialise parameters
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            # predictions using ALL rows
            y_hat = np.dot(x_train, self.coef_) + self.intercept_
            self.loss_history_.append(np.mean((y_train - y_hat) ** 2))

            # gradients (mean over the whole dataset)
            slope_intercept = -2 * np.mean(y_train - y_hat)
            coef_der = -2 * np.dot((y_train - y_hat), x_train) / x_train.shape[0]

            # one update per epoch
            self.intercept_ = self.intercept_ - (self.lr * slope_intercept)
            self.coef_ = self.coef_ - (self.lr * coef_der)

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

🏃 **Train it.** The diabetes features are tiny (around ±0.1), so gradients are tiny too — that is why a relatively large learning rate (`0.4`) and many epochs (`1000`) are needed.

In [ ]:
gdr = GDRegressor(epochs=1000, learning_rate=0.4)
gdr.fit(x_train, y_train)

r2_bgd = r2_score(y_test, gdr.predict(x_test))
print("\nBatch GD R² :", r2_bgd)

✅ Our Batch GD reaches an R² close to the exact `LinearRegression` baseline. The coefficients are *not* identical to `reg.coef_`: with tiny feature values and a finite number of epochs, gradient descent has not fully converged to the exact solution. Stopping early acts like a mild regularizer, which is why the test R² can even come out slightly higher than the baseline. With many more epochs the coefficients move towards the exact ones.

---
## 🎲 6. Stochastic Gradient Descent (SGD)

### 📖 Definition
**Stochastic Gradient Descent** updates the parameters after looking at **just ONE randomly chosen training sample**. If there are `n` samples, one epoch makes about `n` updates.

### 🕒 When is it used?
- Very large datasets (millions of rows)
- Online / streaming learning (data arrives one record at a time)
- Non-convex problems such as neural networks, where noise helps escape bad minima

### ✅ Advantages
- ⚡ Very fast updates and low memory use
- 🌊 Noise can help jump out of shallow local minima
- 🔄 Supports online learning
- 🚀 Often reaches a good solution after seeing only a few passes of the data

### ❌ Disadvantages
- 🎢 Very noisy, zig-zag path — loss does not decrease smoothly
- 🎯 Never settles exactly at the minimum; it keeps bouncing around it
- 🎛️ Sensitive to learning rate (often needs a decaying schedule)
- 🐢 Cannot fully exploit vectorized / GPU hardware (one sample at a time)

### 🧑‍💻 Our own `SGDRegressor` — what is different from Batch GD?
Only **the inner loop**. The parts that are the same as `GDRegressor` (init, predict) are not explained again.

| Line | What it does |
|---|---|
| `for j in range(x_train.shape[0])` | Inner loop → `n` updates per epoch |
| `idx = np.random.randint(0, n)` | Pick **one random row** (sampling with replacement) |
| `y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_` | Prediction for that single row |
| `slope_intercept = -2 * (y_train[idx] - y_hat)` | Gradient from a single sample — **no averaging** needed |
| `coef_der = -2 * (y_train[idx] - y_hat) * x_train[idx]` | One-row gradient for all coefficients |

📉 At the end of every epoch we store the full-data MSE in `loss_history_` — this is only for plotting and does not affect the training.

In [ ]:
class SGDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.loss_history_ = []

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            for j in range(x_train.shape[0]):
                idx = np.random.randint(0, x_train.shape[0])          # ONE random sample
                y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_

                slope_intercept = -2 * (y_train[idx] - y_hat)
                coef_der = -2 * np.dot((y_train[idx] - y_hat), x_train[idx])

                self.intercept_ = self.intercept_ - (self.lr * slope_intercept)
                self.coef_ = self.coef_ - (self.lr * coef_der)

            # (for plotting only) full-data loss after each epoch
            y_full = np.dot(x_train, self.coef_) + self.intercept_
            self.loss_history_.append(np.mean((y_train - y_full) ** 2))

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

In [ ]:
sgd = SGDRegressor(learning_rate=0.01, epochs=50)
sgd.fit(x_train, y_train)

r2_sgd = r2_score(y_test, sgd.predict(x_test))
print("\nOwn SGD R² :", r2_sgd)

### 🧰 scikit-learn's `SGDRegressor` (for comparison)
- `learning_rate='constant'` → keep the step size fixed at `eta0`
- `max_iter` → number of epochs
- `tol=None` → do **not** stop early. (Without it scikit-learn raises a `ConvergenceWarning` when it hits `max_iter` before its tolerance criterion is met.)

In [ ]:
sk_sgd = SklearnSGDRegressor(max_iter=100, learning_rate="constant", eta0=0.01,
                             tol=None, random_state=42)
sk_sgd.fit(x_train, y_train)

r2_sk_sgd = r2_score(y_test, sk_sgd.predict(x_test))
print("scikit-learn SGD R² :", r2_sk_sgd)

---
## 📦 7. Mini-Batch Gradient Descent

### 📖 Definition
**Mini-Batch Gradient Descent** is the *golden middle*. It computes the gradient on a **small random group of samples** (the *batch*, e.g. 32, 64, 128) and updates the parameters after each batch.

### 🕒 When is it used?
- ✅ **The default choice in deep learning** and most large-scale ML
- When you want both speed and stability
- When training on GPUs (batches map perfectly to parallel hardware)

### ✅ Advantages
- ⚖️ Balances the stability of Batch GD with the speed of SGD
- 🚀 Efficient vectorized computation (CPU/GPU friendly)
- 🌊 Some noise → can escape shallow local minima
- 💾 Does not need the entire dataset in memory at once

### ❌ Disadvantages
- 🎛️ Adds an extra hyper-parameter: **batch size**
- 🎢 Still has some oscillation around the minimum
- 🔧 Needs tuning of both batch size and learning rate

### 🧑‍💻 Our own `MBGDRegressor` — what is new?

| Line | What it does |
|---|---|
| `range(int(n / batch_size))` | Number of batches per epoch (e.g. 353 rows / 35 ≈ 10 updates) |
| `idx = random.sample(range(n), batch_size)` | Pick `batch_size` **distinct** random row indices |
| `x_train[idx]`, `y_train[idx]` | Select only that mini-batch |
| `np.mean(...)` and `... / batch_size` | **Average** the gradient over the batch (see note in Section 9) |

Everything else is identical to Batch GD — it is simply Batch GD applied to a *small random subset* instead of the full data.

In [ ]:
class MBGDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100, batch_size=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.loss_history_ = []

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            for j in range(int(x_train.shape[0] / self.batch_size)):
                idx = random.sample(range(x_train.shape[0]), self.batch_size)   # random mini-batch
                y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_

                slope_intercept = -2 * np.mean(y_train[idx] - y_hat)
                coef_der = -2 * np.dot((y_train[idx] - y_hat), x_train[idx]) / self.batch_size

                self.intercept_ = self.intercept_ - (self.lr * slope_intercept)
                self.coef_ = self.coef_ - (self.lr * coef_der)

            # (for plotting only) full-data loss after each epoch
            y_full = np.dot(x_train, self.coef_) + self.intercept_
            self.loss_history_.append(np.mean((y_train - y_full) ** 2))

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

In [ ]:
mbr = MBGDRegressor(batch_size=int(x_train.shape[0] / 10), learning_rate=0.4, epochs=50)
mbr.fit(x_train, y_train)

r2_mbgd = r2_score(y_test, mbr.predict(x_test))
print("\nOwn Mini-Batch GD R² :", r2_mbgd)

### 🧰 Mini-batch with scikit-learn: `partial_fit`
scikit-learn has no dedicated "mini-batch GD" class. The usual way is to call **`partial_fit`** repeatedly, feeding one batch at a time (this is also how out-of-core / streaming learning is done).

ℹ️ Internally, `partial_fit` still performs per-sample SGD updates over the batch it receives, so this is mini-batch *data feeding* rather than a textbook averaged-gradient mini-batch step.

In [ ]:
sk_mb = SklearnSGDRegressor(learning_rate="constant", eta0=0.1, random_state=42)

batch_size = 35
for i in range(100):
    idx = random.sample(range(x_train.shape[0]), batch_size)
    sk_mb.partial_fit(x_train[idx], y_train[idx])

r2_sk_mb = r2_score(y_test, sk_mb.predict(x_test))
print("Coefficients:", sk_mb.coef_)
print("Intercept   :", sk_mb.intercept_)
print("scikit-learn partial_fit R² :", r2_sk_mb)

---
## ⚖️ 8. Comparison of all three variants

### 📊 Conceptual comparison

| Feature 🔍 | ⚡ Batch GD | 🎲 Stochastic GD | 📦 Mini-Batch GD |
|---|---|---|---|
| **Data used per update** | Entire dataset (`n`) | 1 sample | Small batch (e.g. 32–256) |
| **Updates per epoch** | 1 | `n` | `n / batch_size` |
| **Speed per update** | 🐌 Slow | ⚡ Very fast | 🚀 Fast |
| **Convergence path** | Smooth & stable 📉 | Very noisy 🎢 | Slightly noisy 〰️ |
| **Final convergence** | Exact minimum 🎯 | Oscillates around minimum | Close to minimum |
| **Memory need** | High (all data) 💾 | Very low | Moderate |
| **Escapes local minima?** | ❌ Hard | ✅ Easy | ✅ Somewhat |
| **Vectorization / GPU** | ✅ Great | ❌ Poor | ✅ Great |
| **Online learning** | ❌ No | ✅ Yes | ✅ Yes |
| **Extra hyper-parameter** | – | – | Batch size |
| **Best for** | Small, convex problems | Huge data / streaming | 🏆 Deep learning & most real-world tasks |

### 🧪 Experimental comparison on the Diabetes dataset
This table is generated from the models we trained above.

In [ ]:
results = pd.DataFrame({
    "Model": ["LinearRegression (exact)",
              "⚡ Batch GD (own)",
              "🎲 SGD (own)",
              "🎲 SGD (scikit-learn)",
              "📦 Mini-Batch GD (own)",
              "📦 Mini-Batch (sklearn partial_fit)"],
    "R² on test set": [r2_ols, r2_bgd, r2_sgd, r2_sk_sgd, r2_mbgd, r2_sk_mb],
})
results["R² on test set"] = results["R² on test set"].round(4)
results

### 📉 Convergence plot
The plot shows the training MSE after every epoch (log scale on the y-axis). Notice how each variant behaves:
- ⚡ Batch GD → smooth curve, but only **one** update per epoch
- 🎲 SGD → drops *very fast* in the first epochs (many updates per epoch) and then jitters
- 📦 Mini-Batch → in between

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gdr.loss_history_[:50], label="Batch GD",      lw=2)
plt.plot(sgd.loss_history_,      label="Stochastic GD", lw=2)
plt.plot(mbr.loss_history_,      label="Mini-Batch GD", lw=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Training MSE (log scale)")
plt.title("Convergence of the three variants (first 50 epochs)")
plt.legend(); plt.show()

## 📝 9. Notes on code improvements

While organising the notebook, a few small issues in the original code were fixed. Good to know for the future! 💡

| # | Original | Problem | Fix |
|---|---|---|---|
| 1 | `self.b = self.b = (self.lr*loss_slope)` | Typo — assigned instead of subtracting, so `b` was never updated correctly | `self.b -= self.lr * slope_b` |
| 2 | `self.m = m - (...)` | Used the **global** variable `m` instead of `self.m` | Use `self.m` |
| 3 | `coef_der` in Mini-Batch had no `/ batch_size` | Intercept used the *mean* but coefficients used the *sum* → inconsistent gradients | Divide by `batch_size` (average) |
| 4 | `from sklearn.linear_model import SGDRegressor` after writing own `SGDRegressor` | The scikit-learn class **overwrote** your own class | Imported with alias `SklearnSGDRegressor` |
| 5 | `ConvergenceWarning` from scikit-learn SGD | Stopped at `max_iter` before reaching tolerance | `tol=None` |
| 6 | Mini-Batch used `learning_rate=0.01` with the summed gradient | After switching to the averaged gradient (fix 3) the same step size is too small, so we use `0.4` like Batch GD | `learning_rate=0.4` |
| 7 | No random seeds | Results changed on every run | `np.random.seed`, `random.seed`, `random_state` |

🔧 Added: `loss_history_` in every class (for the convergence plots).  
🔎 Simplification to remember: our SGD picks a random row each step *with replacement*, and Mini-Batch draws a fresh random batch each step, so an "epoch" is approximately (not exactly) one pass over the data. Shuffling once per epoch and slicing batches would make it exact.

---
## 🏁 10. Summary

🔑 **Key takeaways**

1. 🏔️ Gradient Descent minimizes a loss by repeatedly stepping **opposite to the gradient**: `θ = θ − η · ∂L/∂θ`.
2. ⚡ **Batch GD** uses all data per update → smooth and stable, but slow on big data.
3. 🎲 **Stochastic GD** uses one sample per update → fast and memory-light, but noisy.
4. 📦 **Mini-Batch GD** uses small batches → best balance, the standard in modern ML/deep learning.
5. 🎛️ The **learning rate** is the most important hyper-parameter: too small is slow, too large diverges.
6. 📏 Scaling the features usually makes gradient descent converge faster (the diabetes data is already scaled).
7. ✅ All three of our scratch implementations reach an R² close to the exact `LinearRegression` solution.

🧭 **Quick guide — which one should I use?**

| Situation | Choice |
|---|---|
| Small dataset, convex problem | ⚡ Batch GD |
| Huge dataset or streaming data | 🎲 SGD |
| Deep learning / GPU training / not sure | 📦 Mini-Batch GD |

🚀 **What to explore next:** Momentum, RMSProp, Adam (smarter optimizers built on top of these ideas), learning-rate schedules, and regularization (Ridge / Lasso).

---
*Happy learning! 🎉*